# NBA Draft Outcome Predictor Modeling Notebook

**Goal**: Classify draft picks into: bust, rotation, starter, and star  
**Model**: XGBoost multiclass classifier  
**Data**: `nba.player_features` : one row per drafted player (2000–2018)

### Sections
1. Load & inspect
2. Preprocessing (imputation, encoding, train/test split)
3. Baseline model
4. Hyperparameter tuning
5. Evaluation (confusion matrix, per-class metrics)
6. Feature importance
7. Backtesting on known draft classes
8. Predict the 2025 draft class

In [ ]:
# !pip install xgboost scikit-learn pandas sqlalchemy psycopg2-binary matplotlib seaborn shap

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import shap

from sqlalchemy import create_engine
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_auc_score
)
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier

pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', '{:.3f}'.format)

RANDOM_STATE = 42
DB_URL = 'postgresql://localhost/nba_draft'   # ← update if needed

---
## 1. Load & Inspect

In [ ]:
engine = create_engine(DB_URL)

df = pd.read_sql("""
    SELECT
        pf.*,
        p.full_name,
        p.position
    FROM nba.player_features pf
    JOIN nba.players p USING (player_id)
    ORDER BY pf.draft_year, pf.pick_overall
""", engine)

print(f"Shape: {df.shape}")
df.head()

In [ ]:
# Outcome distribution:  Checking for class distribution
outcome_counts = df['outcome'].value_counts()
print(outcome_counts)
print()
print(outcome_counts / len(df) * 100)

In [ ]:
# Null rates per column
# Combine cols (~40-50% null) are expected from players who skipped the combine
# College cols should be low null, flags anything above 10%
null_rates = df.isnull().mean().sort_values(ascending=False)
null_rates[null_rates > 0].plot(
    kind='barh', figsize=(8, 6), title='Null Rate by Feature'
)
plt.axvline(0.1, color='red', linestyle='--', label='10% threshold')
plt.xlabel('Null rate')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Feature distributions by outcome
# Stars should cluster at low pick numbers and high BPM
key_features = ['pick_overall', 'age_at_draft', 'adj_pts_per36', 'bpm', 'ts_pct', 'win_shares']

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
outcome_order = ['star', 'starter', 'rotation', 'bust']
palette = {'star': '#FFD700', 'starter': '#4CAF50', 'rotation': '#2196F3', 'bust': '#F44336'}

for ax, feat in zip(axes.flat, key_features):
    sns.boxplot(
        data=df, x='outcome', y=feat,
        order=outcome_order, palette=palette, ax=ax
    )
    ax.set_title(feat)
    ax.set_xlabel('')

plt.suptitle('Feature Distributions by Outcome', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 2. Preprocessing

In [ ]:
# Feature columns, Split into two groups:
#   COLLEGE_FEATURES  — always present (required for row inclusion)
#   COMBINE_FEATURES  — often null; imputed with column median
# We also add a binary flag for each combine feature to let the
# model learn that 'no combine attendance' is itself a signal.

COLLEGE_FEATURES = [
    'pick_overall',
    'age_at_draft',
    'adj_pts_per36',
    'adj_reb_per36',
    'adj_ast_per36',
    'ast_to_ratio',
    'ts_pct',
    'usg_pct',
    'win_shares',
    'bpm',
    'conference_tier',
    'pts_yoy_delta',
    'win_shares_yoy_delta',
]

COMBINE_FEATURES = [
    'wingspan_height_ratio',
    'max_vertical',
    'lane_agility_time',
    'body_fat_pct',
]

TARGET = 'outcome'
OUTCOME_ORDER = ['bust', 'rotation', 'starter', 'star']  # ordinal-ish order

In [ ]:
# Add missingness indicator flags for combine features
# XGBoost can use 'did not attend combine' as a feature itself
for col in COMBINE_FEATURES:
    df[f'{col}_missing'] = df[col].isnull().astype(int)

MISSING_FLAGS = [f'{c}_missing' for c in COMBINE_FEATURES]
ALL_FEATURES = COLLEGE_FEATURES + COMBINE_FEATURES + MISSING_FLAGS

print(f"Total features: {len(ALL_FEATURES)}")
print(ALL_FEATURES)

In [ ]:
# Encode target
le = LabelEncoder()
le.fit(OUTCOME_ORDER)
df['outcome_encoded'] = le.transform(df[TARGET])

print("Label mapping:")
for cls, idx in zip(le.classes_, le.transform(le.classes_)):
    print(f"  {idx} → {cls}")

In [1]:
# Train / Test split
# Strategy: hold out the most recent 3 draft classes as test set
# This simulates real-world usage — you train on past classes,
# predict on incoming ones. Avoids data leakage from random splits.
HOLDOUT_YEARS = [2016, 2017, 2018]

train_df = df[~df['draft_year'].isin(HOLDOUT_YEARS)].copy()
test_df  = df[ df['draft_year'].isin(HOLDOUT_YEARS)].copy()

print(f"Train: {len(train_df)} players ({train_df['draft_year'].min()}–{train_df['draft_year'].max()})")
print(f"Test:  {len(test_df)}  players ({test_df['draft_year'].min()}–{test_df['draft_year'].max()})")

X_train = train_df[ALL_FEATURES]
y_train = train_df['outcome_encoded']
X_test  = test_df[ALL_FEATURES]
y_test  = test_df['outcome_encoded']

NameError: name 'df' is not defined

In [ ]:
# Impute combine nulls with column median (fit on train only)
imputer = SimpleImputer(strategy='median')
X_train_imp = pd.DataFrame(
    imputer.fit_transform(X_train),
    columns=ALL_FEATURES
)
X_test_imp = pd.DataFrame(
    imputer.transform(X_test),
    columns=ALL_FEATURES
)

## 3. Baseline Model

In [ ]:
# Baseline XGBoost with sensible defaults
# We use scale_pos_weight implicitly via sample_weight;
# XGBoost handles class imbalance reasonably with
# eval_metric='mlogloss' but we'll tune further below.
baseline = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softprob',
    num_class=4,
    eval_metric='mlogloss',
    random_state=RANDOM_STATE,
    verbosity=0,
)

baseline.fit(X_train_imp, y_train)

y_pred = baseline.predict(X_test_imp)
y_prob = baseline.predict_proba(X_test_imp)

print("=== Baseline Classification Report ===")
print(classification_report(y_test, y_pred, target_names=le.classes_))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
fig, ax = plt.subplots(figsize=(7, 6))
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Baseline Confusion Matrix (holdout 2016–2018)')
plt.tight_layout()
plt.show()

# Note: Most misclassifications will be adjacent classes (rotation↔starter).
# Bust↔Star confusion is the real failure mode to watch.

In [ ]:
# Cross-val on training set — confirms we're not just lucky on holdout
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(baseline, X_train_imp, y_train, cv=cv, scoring='accuracy')

print(f"5-fold CV accuracy: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

## 4. Hyperparameter Tuning

In [ ]:
# RandomizedSearchCV — faster than GridSearch for XGBoost
# Run this once; once you find good params, hardcode them below
param_dist = {
    'n_estimators':      [100, 200, 300, 500],
    'max_depth':         [3, 4, 5, 6],
    'learning_rate':     [0.01, 0.03, 0.05, 0.1],
    'subsample':         [0.6, 0.7, 0.8, 0.9],
    'colsample_bytree':  [0.6, 0.7, 0.8, 1.0],
    'min_child_weight':  [1, 3, 5],
    'gamma':             [0, 0.1, 0.3, 0.5],
    'reg_alpha':         [0, 0.1, 0.5, 1.0],   # L1
    'reg_lambda':        [0.5, 1.0, 2.0, 5.0], # L2
}

xgb_base = XGBClassifier(
    objective='multi:softprob',
    num_class=4,
    eval_metric='mlogloss',
    random_state=RANDOM_STATE,
    verbosity=0,
)

search = RandomizedSearchCV(
    xgb_base,
    param_distributions=param_dist,
    n_iter=50,              # increase to 100+ for more thorough search
    scoring='accuracy',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
)

search.fit(X_train_imp, y_train)

print(f"Best CV accuracy: {search.best_score_:.3f}")
print(f"Best params:\n{search.best_params_}")

In [ ]:
# Refit best model and evaluate on holdout
best_model = search.best_estimator_

y_pred_best = best_model.predict(X_test_imp)
y_prob_best = best_model.predict_proba(X_test_imp)

print("=== Tuned Model — Classification Report ===")
print(classification_report(y_test, y_pred_best, target_names=le.classes_))

## 5. Evaluation

In [ ]:
# Confusion matrix — tuned model
cm_best = confusion_matrix(y_test, y_pred_best)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_best, display_labels=le.classes_)
fig, ax = plt.subplots(figsize=(7, 6))
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Tuned Model — Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Multiclass ROC-AUC (one-vs-rest)
auc = roc_auc_score(y_test, y_prob_best, multi_class='ovr', average='macro')
print(f"Macro ROC-AUC (OvR): {auc:.3f}")

# Per-class AUC
for i, cls in enumerate(le.classes_):
    y_bin = (y_test == i).astype(int)
    auc_cls = roc_auc_score(y_bin, y_prob_best[:, i])
    print(f"  {cls:<12} AUC: {auc_cls:.3f}")

In [ ]:
# Probability calibration check
# Plots average predicted probability vs. actual rate per outcome
# A well-calibrated model should hug the diagonal
from sklearn.calibration import calibration_curve

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, (cls, ax) in enumerate(zip(le.classes_, axes)):
    y_bin = (y_test == i).astype(int)
    if y_bin.sum() == 0:
        continue
    prob_true, prob_pred = calibration_curve(y_bin, y_prob_best[:, i], n_bins=8)
    ax.plot(prob_pred, prob_true, marker='o', label='Model')
    ax.plot([0, 1], [0, 1], 'k--', label='Perfect')
    ax.set_title(cls)
    ax.set_xlabel('Mean predicted prob')
    ax.set_ylabel('Fraction positive')
    ax.legend(fontsize=8)

plt.suptitle('Probability Calibration by Outcome Class', y=1.02)
plt.tight_layout()
plt.show()

## 6. Feature Importance

In [ ]:
# Native XGBoost importance (gain — how much each feature reduces loss)
importance_df = pd.DataFrame({
    'feature':    ALL_FEATURES,
    'importance': best_model.feature_importances_
}).sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(8, len(ALL_FEATURES) * 0.35 + 1))
ax.barh(importance_df['feature'], importance_df['importance'], color='steelblue')
ax.set_title('XGBoost Feature Importance (Gain)')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

In [ ]:
# SHAP values — more trustworthy than native importance
# Shows direction of effect, not just magnitude
explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test_imp)

# Summary plot for 'star' class (index 3)
print("SHAP summary — predicting 'star' outcome:")
shap.summary_plot(
    shap_values[3],
    X_test_imp,
    feature_names=ALL_FEATURES,
    plot_type='dot',
    max_display=15,
    show=True
)

In [ ]:
# SHAP summary for 'bust' class (index 0)
print("SHAP summary — predicting 'bust' outcome:")
shap.summary_plot(
    shap_values[0],
    X_test_imp,
    feature_names=ALL_FEATURES,
    plot_type='dot',
    max_display=15,
    show=True
)

## 7. Backtesting on 2003 NBA draft

In [ ]:
def backtest_class(year: int) -> pd.DataFrame:
    """Return predictions + actuals for a single draft class."""
    mask = df['draft_year'] == year
    subset = df[mask].copy()

    X = pd.DataFrame(
        imputer.transform(subset[ALL_FEATURES]),
        columns=ALL_FEATURES
    )
    probs = best_model.predict_proba(X)
    preds = best_model.predict(X)

    result = subset[['full_name', 'pick_overall', 'outcome']].copy().reset_index(drop=True)
    result['predicted']    = le.inverse_transform(preds)
    result['correct']      = result['outcome'] == result['predicted']

    for i, cls in enumerate(le.classes_):
        result[f'p_{cls}'] = np.round(probs[:, i], 3)

    return result.sort_values('pick_overall')


# 2003 — LeBron, Carmelo, Bosh, Wade draft
bt_2003 = backtest_class(2003)
print(f"2003 Draft Class — Accuracy: {bt_2003['correct'].mean():.1%}\n")
bt_2003[['full_name', 'pick_overall', 'outcome', 'predicted', 'p_star', 'p_bust']].head(20)

In [ ]:
# Accuracy by draft year — shows which classes the model struggles with
yearly_acc = []
for year in sorted(df['draft_year'].unique()):
    bt = backtest_class(year)
    yearly_acc.append({'draft_year': year, 'accuracy': bt['correct'].mean(), 'n': len(bt)})

yearly_df = pd.DataFrame(yearly_acc)

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(yearly_df['draft_year'], yearly_df['accuracy'], color='steelblue')
ax.axhline(yearly_df['accuracy'].mean(), color='red', linestyle='--', label=f"Mean: {yearly_df['accuracy'].mean():.1%}")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_title('Prediction Accuracy by Draft Year')
ax.set_xlabel('Draft Year')
ax.set_ylabel('Accuracy')
ax.legend()
plt.tight_layout()
plt.show()

## 8. Predicting the 2025 Draft Class

In [ ]:
# Load 2025 prospects from the database
# These should have been ingested by bref_scraper.py with --end 2025
# and will have NULL outcomes (they haven't played yet)
prospects_2025 = pd.read_sql("""
    SELECT
        pf.*,
        p.full_name,
        p.position
    FROM nba.player_features pf
    JOIN nba.players p USING (player_id)
    WHERE pf.draft_year = 2025
    ORDER BY pf.pick_overall NULLS LAST
""", engine)

print(f"{len(prospects_2025)} prospects loaded")
prospects_2025.head()

In [ ]:
def predict_prospects(prospect_df: pd.DataFrame) -> pd.DataFrame:
    """Run the trained model on a set of prospects without known outcomes."""
    X = pd.DataFrame(
        imputer.transform(prospect_df[ALL_FEATURES]),
        columns=ALL_FEATURES
    )
    probs = best_model.predict_proba(X)
    preds = best_model.predict(X)

    out = prospect_df[['full_name', 'pick_overall', 'position',
                        'adj_pts_per36', 'bpm', 'ts_pct', 'wingspan_height_ratio']].copy().reset_index(drop=True)
    out['predicted_outcome'] = le.inverse_transform(preds)

    for i, cls in enumerate(le.classes_):
        out[f'p_{cls}'] = np.round(probs[:, i], 3)

    # Composite star score — useful for ranking prospects
    out['star_score'] = np.round(
        out['p_star'] * 1.0 + out['p_starter'] * 0.5, 3
    )

    return out.sort_values('star_score', ascending=False)


predictions_2025 = predict_prospects(prospects_2025)
print("=== 2025 Draft Class Predictions (ranked by star score) ===")
predictions_2025[[
    'full_name', 'pick_overall', 'predicted_outcome',
    'p_star', 'p_starter', 'p_rotation', 'p_bust', 'star_score'
]].head(30)

In [ ]:
# Visualize the probability distribution for top prospects
top_n = predictions_2025.head(15)

fig, ax = plt.subplots(figsize=(12, 6))
bottom = np.zeros(len(top_n))
colors = {'p_bust': '#F44336', 'p_rotation': '#2196F3',
          'p_starter': '#4CAF50', 'p_star': '#FFD700'}

for col, color in colors.items():
    ax.bar(top_n['full_name'], top_n[col], bottom=bottom,
           label=col.replace('p_', '').title(), color=color)
    bottom += top_n[col].values

ax.set_xticklabels(top_n['full_name'], rotation=45, ha='right')
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_title('2025 Draft Class — Outcome Probability Distribution (Top 15)')
ax.set_ylabel('Probability')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

In [ ]:
# SHAP waterfall for a single prospect — great for explainability
# Change index to look at any player
PROSPECT_IDX = 0  # top-ranked prospect

prospect_name = predictions_2025.iloc[PROSPECT_IDX]['full_name']
X_prospect = pd.DataFrame(
    imputer.transform(prospects_2025[ALL_FEATURES]),
    columns=ALL_FEATURES
).iloc[PROSPECT_IDX:PROSPECT_IDX+1]

shap_prospect = explainer.shap_values(X_prospect)

print(f"SHAP explanation — {prospect_name} — 'star' probability:")
shap.waterfall_plot(
    shap.Explanation(
        values=shap_prospect[3][0],
        base_values=explainer.expected_value[3],
        data=X_prospect.iloc[0],
        feature_names=ALL_FEATURES
    )
)

In [ ]:
# Save predictions to DB for use in Streamlit dashboard
predictions_2025.to_sql(
    'draft_predictions_2025',
    engine,
    schema='nba',
    if_exists='replace',
    index=False
)
print("Predictions saved to nba.draft_predictions_2025")